# Query Dis2Vec Pre-computed Predictions

This notebook queries pre-computed disease-disease association predictions from the Dis2Vec UPNA-DDAL model.

**Data source**: `dis2vec_all_preds.parquet.gzip` from [Zenodo](https://zenodo.org/records/21251215)

**Model performance**: AUROC = 0.89 on IxIDN/ORDON benchmarks

## Prerequisites

Download from Zenodo (https://zenodo.org/records/21251215):
1. `dis2vec_all_preds.parquet.gzip` (244.6 MB) → place in `../data/`
2. `dis2vec_clust.xlsx` (3.8 MB) → place in `../data/` [already present]

## 1. Setup and Load Data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print('Imports successful!')

Imports successful!


In [ ]:
# Paths
DATA_DIR = Path('../data')
PREDS_FILE = DATA_DIR / 'dis2vec_all_preds.parquet.gzip'
CLUST_FILE = DATA_DIR / 'dis2vec_clust.xlsx'

# Check if files exist
if not PREDS_FILE.exists():
    print(f'❌ Missing: {PREDS_FILE}')
    print('   Download from: https://zenodo.org/records/21253352')
    raise FileNotFoundError(f'Please download {PREDS_FILE.name} from Zenodo')

if not CLUST_FILE.exists():
    print(f'⚠️  Missing: {CLUST_FILE} (optional - for disease names)')

print('✅ All required files found!')

✅ All required files found!


In [3]:
# Load pre-computed predictions
print('Loading predictions (this may take a moment)...')
all_preds_df = pd.read_parquet(PREDS_FILE)

print(f'✅ Loaded {len(all_preds_df):,} disease-disease predictions')
print(f'\nColumns: {list(all_preds_df.columns)}')
print(f'\nPreview:')
all_preds_df.head()

Loading predictions (this may take a moment)...
✅ Loaded 11,094,404 disease-disease predictions

Columns: ['pair', 'Dis2vec_Pred_mean', 'Dis2vec_Pred_std', 'node1_disease_id', 'node2_disease_id', 'node1_disease_name', 'node2_disease_name']

Preview:


,pair,Dis2vec_Pred_mean,Dis2vec_Pred_std,node1_disease_id,node2_disease_id,node1_disease_name,node2_disease_name
0,ORPHA_2332__ORPHA_440354,0.787477,0.105730,ORPHA_2332,ORPHA_440354,KBG syndrome,Autosomal dominant myopia-midfacial retrusion-...
1,ORPHA_352596__ORPHA_79096,0.861128,0.007597,ORPHA_79096,ORPHA_352596,Pyridoxal phosphate-responsive seizures,Progressive myoclonic epilepsy with dystonia
2,ORPHA_71529__ORPHA_93267,0.000394,0.000290,ORPHA_93267,ORPHA_71529,Cloverleaf skull-multiple congenital anomalies...,Obesity due to melanocortin 4 receptor deficiency
3,ORPHA_100974__ORPHA_79478,0.170978,0.033091,ORPHA_79478,ORPHA_100974,Griscelli syndrome type 3,FRAXF syndrome
4,ORPHA_1657__ORPHA_90307,0.719929,0.147701,ORPHA_1657,ORPHA_90307,"Dermatoosteolysis, Kirghizian type",Parkes Weber syndrome


In [4]:
# Load disease clustering info (optional - for additional metadata)
if CLUST_FILE.exists():
    clust_df = pd.read_excel(CLUST_FILE)
    print(f'✅ Loaded clustering data for {len(clust_df)} diseases')
    print(f'\nColumns: {list(clust_df.columns)}')
else:
    clust_df = None
    print('⚠️  Clustering data not loaded')

✅ Loaded clustering data for 4710 diseases

Columns: ['Unnamed: 0', 'node', 'dis_vec', 'disease_id', 'name', 'dis_vec_clust', 'PCA', 'UMAP', 'x', 'y', 'clust lbl', 'rare']


## 2. Dataset Statistics

In [5]:
all_preds_df

,pair,Dis2vec_Pred_mean,Dis2vec_Pred_std,node1_disease_id,node2_disease_id,node1_disease_name,node2_disease_name
0,ORPHA_2332__ORPHA_440354,0.787477,0.105730,ORPHA_2332,ORPHA_440354,KBG syndrome,Autosomal dominant myopia-midfacial retrusion-...
1,ORPHA_352596__ORPHA_79096,0.861128,0.007597,ORPHA_79096,ORPHA_352596,Pyridoxal phosphate-responsive seizures,Progressive myoclonic epilepsy with dystonia
2,ORPHA_71529__ORPHA_93267,0.000394,0.000290,ORPHA_93267,ORPHA_71529,Cloverleaf skull-multiple congenital anomalies...,Obesity due to melanocortin 4 receptor deficiency
3,ORPHA_100974__ORPHA_79478,0.170978,0.033091,ORPHA_79478,ORPHA_100974,Griscelli syndrome type 3,FRAXF syndrome
4,ORPHA_1657__ORPHA_90307,0.719929,0.147701,ORPHA_1657,ORPHA_90307,"Dermatoosteolysis, Kirghizian type",Parkes Weber syndrome
...,...,...,...,...,...,...,...
11094399,ORPHA_279882__ORPHA_90342,0.002016,0.000189,ORPHA_279882,ORPHA_90342,Spasmus nutans,Xeroderma pigmentosum variant
11094400,ORPHA_178464__ORPHA_71493,0.007933,0.002401,ORPHA_178464,ORPHA_71493,Hereditary myopathy with early respiratory fai...,Familial thrombocytosis
11094401,EFO_0000274__ORPHA_353298,0.001939,0.000581,ORPHA_353298,EFO_0000274,Roifman syndrome,atopic eczema
11094402,ORPHA_163934__ORPHA_3463,0.033205,0.010691,ORPHA_163934,ORPHA_3463,Atopic keratoconjunctivitis,Wolfram syndrome


In [6]:
print('Dataset Statistics:')
print(f'  Total predictions: {len(all_preds_df):,}')
print(f'  Unique diseases: {pd.concat([all_preds_df["node1_disease_id"], all_preds_df["node2_disease_id"]]).nunique()}')
print(f'\nPrediction score distribution:')
print(all_preds_df['Dis2vec_Pred_mean'].describe())

# Score ranges
print(f'\nScore ranges:')
print(f'  High confidence (>0.8): {(all_preds_df["Dis2vec_Pred_mean"] > 0.8).sum():,} ({(all_preds_df["Dis2vec_Pred_mean"] > 0.8).sum() / len(all_preds_df) * 100:.2f}%)')
print(f'  Medium (0.5-0.8): {((all_preds_df["Dis2vec_Pred_mean"] >= 0.5) & (all_preds_df["Dis2vec_Pred_mean"] <= 0.8)).sum():,} ({((all_preds_df["Dis2vec_Pred_mean"] >= 0.5) & (all_preds_df["Dis2vec_Pred_mean"] <= 0.8)).sum() / len(all_preds_df) * 100:.2f}%)')
print(f'  Low (<0.5): {(all_preds_df["Dis2vec_Pred_mean"] < 0.5).sum():,} ({(all_preds_df["Dis2vec_Pred_mean"] < 0.5).sum() / len(all_preds_df) * 100:.2f}%)')

Dataset Statistics:
  Total predictions: 11,094,404
  Unique diseases: 4710

Prediction score distribution:
count    1.109440e+07
mean     2.070901e-01
std      3.440444e-01
min      1.174580e-07
25%      3.928139e-04
50%      6.942877e-03
75%      2.547339e-01
max      9.999990e-01
Name: Dis2vec_Pred_mean, dtype: float64

Score ranges:
  High confidence (>0.8): 1,549,695 (13.97%)
  Medium (0.5-0.8): 638,418 (5.75%)
  Low (<0.5): 8,906,291 (80.28%)


## 3. Query Functions

In [7]:
def query_disease_pair(disease_id_1, disease_id_2, df=all_preds_df):
    """
    Query prediction for a specific disease pair.
    
    Args:
        disease_id_1: First disease ID (e.g., 'ORPHA_2134')
        disease_id_2: Second disease ID (e.g., 'ORPHA_447')
        df: Predictions dataframe
    
    Returns:
        DataFrame with prediction(s) for this pair
    """
    # Create canonical pair ID (sorted)
    pair_id = '__'.join(sorted([disease_id_1, disease_id_2]))
    
    result = df[df['pair'] == pair_id]
    
    if len(result) == 0:
        print(f'⚠️  No predictions found for pair: {disease_id_1} <-> {disease_id_2}')
        return None
    
    return result


def get_top_associations(disease_id, top_n=20, df=all_preds_df):
    """
    Get top N predicted associations for a disease.
    
    Args:
        disease_id: Disease ID (e.g., 'ORPHA_2134')
        top_n: Number of top associations to return
        df: Predictions dataframe
    
    Returns:
        DataFrame sorted by prediction score (descending)
    """
    # Find all rows where disease appears
    mask = (df['node1_disease_id'] == disease_id) | (df['node2_disease_id'] == disease_id)
    results = df[mask].copy()
    
    if len(results) == 0:
        print(f'⚠️  No predictions found for disease: {disease_id}')
        return None
    
    # Identify the "other" disease in each pair
    results['target_disease'] = disease_id
    results['associated_disease_id'] = results.apply(
        lambda x: x['node2_disease_id'] if x['node1_disease_id'] == disease_id else x['node1_disease_id'],
        axis=1
    )
    results['associated_disease_name'] = results.apply(
        lambda x: x['node2_disease_name'] if x['node1_disease_id'] == disease_id else x['node1_disease_name'],
        axis=1
    )
    
    # Sort by prediction score and return top N
    results = results.sort_values('Dis2vec_Pred_mean', ascending=False).head(top_n)
    
    return results[['associated_disease_id', 'associated_disease_name', 'Dis2vec_Pred_mean', 'pair']]


def search_disease_by_name(name_substring, df=all_preds_df):
    """
    Search for diseases by name substring.
    
    Args:
        name_substring: Partial disease name to search for
        df: Predictions dataframe
    
    Returns:
        DataFrame with matching diseases and their IDs
    """
    # Search in both node1 and node2 disease names
    mask1 = df['node1_disease_name'].str.contains(name_substring, case=False, na=False)
    mask2 = df['node2_disease_name'].str.contains(name_substring, case=False, na=False)
    
    matches1 = df[mask1][['node1_disease_id', 'node1_disease_name']].drop_duplicates()
    matches2 = df[mask2][['node2_disease_id', 'node2_disease_name']].drop_duplicates()
    
    matches1.columns = ['disease_id', 'disease_name']
    matches2.columns = ['disease_id', 'disease_name']
    
    all_matches = pd.concat([matches1, matches2]).drop_duplicates().sort_values('disease_name')
    
    return all_matches.reset_index(drop=True)


print('Query functions defined!')

Query functions defined!


## 4. Example Queries

### Example 1: Search for a disease by name

In [8]:
# Search for diseases related to 'hemolytic'
search_results = search_disease_by_name('hemolytic')
print(f'Found {len(search_results)} diseases matching "hemolytic":\n')
search_results.head(10)

Found 12 diseases matching "hemolytic":



,disease_id,disease_name
0,ORPHA_2134,Atypical hemolytic uremic syndrome
1,ORPHA_93581,Atypical hemolytic uremic syndrome with anti-f...
2,ORPHA_544472,Atypical hemolytic uremic syndrome with comple...
3,ORPHA_90033,"Autoimmune hemolytic anemia, warm type"
4,ORPHA_444463,Autoimmune hemolytic anemia-autoimmune thrombo...
5,ORPHA_90037,Drug-induced autoimmune hemolytic anemia
6,ORPHA_766,Hemolytic anemia due to red cell pyruvate kina...
7,ORPHA_544482,Infection-related hemolytic uremic syndrome
8,ORPHA_1046,Lethal hemolytic anemia-genital anomalies synd...
9,ORPHA_90036,Mixed-type autoimmune hemolytic anemia


### Example 2: Query specific disease pair

**Classic example from manuscript**: aHUS (ORPHA_2134) ↔ PNH (ORPHA_447)

In [9]:
# Query aHUS <-> PNH association (from manuscript)
ahus_pnh = query_disease_pair('ORPHA_2134', 'ORPHA_447')

if ahus_pnh is not None:
    print('aHUS ↔ PNH Association:')
    print(f"  Prediction Score: {ahus_pnh['Dis2vec_Pred_mean'].values[0]:.4f}")
    print(f"  Disease 1: {ahus_pnh['node1_disease_name'].values[0]} ({ahus_pnh['node1_disease_id'].values[0]})")
    print(f"  Disease 2: {ahus_pnh['node2_disease_name'].values[0]} ({ahus_pnh['node2_disease_id'].values[0]})")
    print(f"\n  High score (>0.8) indicates strong predicted therapeutic transferability")
    display(ahus_pnh)

aHUS ↔ PNH Association:
  Prediction Score: 0.9937
  Disease 1: Atypical hemolytic uremic syndrome (ORPHA_2134)
  Disease 2: Paroxysmal nocturnal hemoglobinuria (ORPHA_447)

  High score (>0.8) indicates strong predicted therapeutic transferability


,pair,Dis2vec_Pred_mean,Dis2vec_Pred_std,node1_disease_id,node2_disease_id,node1_disease_name,node2_disease_name
2146503,ORPHA_2134__ORPHA_447,0.993673,0.001072,ORPHA_2134,ORPHA_447,Atypical hemolytic uremic syndrome,Paroxysmal nocturnal hemoglobinuria


### Example 3: Find top associations for a disease

In [10]:
# Find top 20 associations for aHUS
ahus_id = 'ORPHA_2134'
top_assoc = get_top_associations(ahus_id, top_n=20)

if top_assoc is not None:
    print(f'Top 20 predicted associations for Atypical Hemolytic Uremic Syndrome (aHUS):\n')
    display(top_assoc)

Top 20 predicted associations for Atypical Hemolytic Uremic Syndrome (aHUS):



,associated_disease_id,associated_disease_name,Dis2vec_Pred_mean,pair
2519168,ORPHA_2134,Atypical hemolytic uremic syndrome,0.999071,ORPHA_2134__ORPHA_2134
4507178,ORPHA_93581,Atypical hemolytic uremic syndrome with anti-f...,0.999057,ORPHA_2134__ORPHA_93581
2621152,MONDO_0008691,"zinc, elevated plasma",0.998441,MONDO_0008691__ORPHA_2134
7591109,ORPHA_275555,Preeclampsia,0.998083,ORPHA_2134__ORPHA_275555
1510888,ORPHA_103909,Trehalase deficiency,0.997642,ORPHA_103909__ORPHA_2134
6554593,ORPHA_544472,Atypical hemolytic uremic syndrome with comple...,0.996973,ORPHA_2134__ORPHA_544472
3495385,ORPHA_98827,Unclassified myelodysplastic syndrome,0.996880,ORPHA_2134__ORPHA_98827
1768328,ORPHA_859,Transcobalamin deficiency,0.996801,ORPHA_2134__ORPHA_859
2256176,ORPHA_488,Urachal cyst,0.996656,ORPHA_2134__ORPHA_488
5067148,ORPHA_98870,Congenital dyserythropoietic anemia type III,0.996503,ORPHA_2134__ORPHA_98870


## 5. Manuscript Case Studies

These examples are highlighted in the manuscript.

In [11]:
# Case Study: Rheumatoid Arthritis <-> Idiopathic Inflammatory Myopathies
ra_id = 'EFO_0000685'  # Rheumatoid Arthritis

iim_diseases = [
    ('ORPHA_81', 'Antisynthetase Syndrome'),
    ('ORPHA_732', 'Polymyositis'),
    ('ORPHA_645613', 'Dermatomyositis'),
    ('ORPHA_809', 'Mixed Connective Tissue Disease'),
    ('ORPHA_206572', 'Overlap Myositis')
]

print('Rheumatoid Arthritis <-> Idiopathic Inflammatory Myopathies:\n')
print('Demonstrates rare-to-non-rare therapeutic transfer prediction\n')

ra_iim_results = []
for iim_id, iim_name in iim_diseases:
    result = query_disease_pair(ra_id, iim_id)
    if result is not None and len(result) > 0:
        score = result['Dis2vec_Pred_mean'].values[0]
        ra_iim_results.append({
            'IIM_Disease': iim_name,
            'ORPHA_ID': iim_id,
            'Dis2vec_Score': score
        })

ra_iim_df = pd.DataFrame(ra_iim_results).sort_values('Dis2vec_Score', ascending=False)
display(ra_iim_df)

Rheumatoid Arthritis <-> Idiopathic Inflammatory Myopathies:

Demonstrates rare-to-non-rare therapeutic transfer prediction



,IIM_Disease,ORPHA_ID,Dis2vec_Score
0,Antisynthetase Syndrome,ORPHA_81,0.983776
3,Mixed Connective Tissue Disease,ORPHA_809,0.982725
4,Overlap Myositis,ORPHA_206572,0.972003
2,Dermatomyositis,ORPHA_645613,0.949287
1,Polymyositis,ORPHA_732,0.930126


## 6. Custom Queries

Use this section for your own disease pair queries.

In [12]:
# Custom query 1: Search for a disease
search_term = 'parkinson'  # Change this
results = search_disease_by_name(search_term)
print(f'Diseases matching "{search_term}":')
display(results.head(10))

Diseases matching "parkinson":


,disease_id,disease_name
0,ORPHA_199351,Adult-onset dystonia-parkinsonism
1,ORPHA_391411,Atypical juvenile parkinsonism
2,ORPHA_97355,Caribbean parkinsonism
3,ORPHA_306692,Cyanide-induced parkinsonism-dystonia
4,ORPHA_521406,Dystonia-parkinsonism-hypermanganesemia syndrome
5,ORPHA_2379,Early-onset parkinsonism-intellectual disabili...
6,ORPHA_306669,Hemiparkinsonism-hemiatrophy syndrome
7,ORPHA_411602,Hereditary late-onset Parkinson disease
8,ORPHA_238455,Infantile dystonia-parkinsonism
9,ORPHA_98933,"Multiple system atrophy, parkinsonian type"


In [13]:
# Custom query 2: Specific disease pair
disease_1_id = 'ORPHA_2134'  # Change these IDs
disease_2_id = 'ORPHA_447'

result = query_disease_pair(disease_1_id, disease_2_id)
if result is not None:
    display(result)

,pair,Dis2vec_Pred_mean,Dis2vec_Pred_std,node1_disease_id,node2_disease_id,node1_disease_name,node2_disease_name
2146503,ORPHA_2134__ORPHA_447,0.993673,0.001072,ORPHA_2134,ORPHA_447,Atypical hemolytic uremic syndrome,Paroxysmal nocturnal hemoglobinuria


In [14]:
# Custom query 3: Top associations
target_disease_id = 'ORPHA_2134'  # Change this
n_results = 30  # Number of top results

top_results = get_top_associations(target_disease_id, top_n=n_results)
if top_results is not None:
    print(f'Top {n_results} associations:')
    display(top_results)

Top 30 associations:


,associated_disease_id,associated_disease_name,Dis2vec_Pred_mean,pair
2519168,ORPHA_2134,Atypical hemolytic uremic syndrome,0.999071,ORPHA_2134__ORPHA_2134
4507178,ORPHA_93581,Atypical hemolytic uremic syndrome with anti-f...,0.999057,ORPHA_2134__ORPHA_93581
2621152,MONDO_0008691,"zinc, elevated plasma",0.998441,MONDO_0008691__ORPHA_2134
7591109,ORPHA_275555,Preeclampsia,0.998083,ORPHA_2134__ORPHA_275555
1510888,ORPHA_103909,Trehalase deficiency,0.997642,ORPHA_103909__ORPHA_2134
6554593,ORPHA_544472,Atypical hemolytic uremic syndrome with comple...,0.996973,ORPHA_2134__ORPHA_544472
3495385,ORPHA_98827,Unclassified myelodysplastic syndrome,0.996880,ORPHA_2134__ORPHA_98827
1768328,ORPHA_859,Transcobalamin deficiency,0.996801,ORPHA_2134__ORPHA_859
2256176,ORPHA_488,Urachal cyst,0.996656,ORPHA_2134__ORPHA_488
5067148,ORPHA_98870,Congenital dyserythropoietic anemia type III,0.996503,ORPHA_2134__ORPHA_98870


## 7. Advanced Filtering

Filter predictions by score threshold or other criteria.

In [17]:
# Find all high-confidence predictions (>0.9) for a disease
disease_id = 'ORPHA_2134'
threshold = 0.9

mask = ((all_preds_df['node1_disease_id'] == disease_id) | 
        (all_preds_df['node2_disease_id'] == disease_id)) & \
       (all_preds_df['Dis2vec_Pred_mean'] >= threshold)

high_conf = all_preds_df[mask].copy()

# Add 'other disease' column
high_conf['associated_disease_id'] = high_conf.apply(
    lambda x: x['node2_disease_id'] if x['node1_disease_id'] == disease_id else x['node1_disease_id'],
    axis=1
)
high_conf['associated_disease_name'] = high_conf.apply(
    lambda x: x['node2_disease_name'] if x['node1_disease_id'] == disease_id else x['node1_disease_name'],
    axis=1
)

high_conf = high_conf.sort_values('Dis2vec_Pred_mean', ascending=False)

print(f'High-confidence associations (score ≥ {threshold}) for {disease_id}:')
print(f'Found {len(high_conf)} associations\n')
display(high_conf[['associated_disease_id', 'associated_disease_name', 'Dis2vec_Pred_mean']].head(20))

High-confidence associations (score ≥ 0.9) for ORPHA_2134:
Found 538 associations



,associated_disease_id,associated_disease_name,Dis2vec_Pred_mean
2519168,ORPHA_2134,Atypical hemolytic uremic syndrome,0.999071
4507178,ORPHA_93581,Atypical hemolytic uremic syndrome with anti-f...,0.999057
2621152,MONDO_0008691,"zinc, elevated plasma",0.998441
7591109,ORPHA_275555,Preeclampsia,0.998083
1510888,ORPHA_103909,Trehalase deficiency,0.997642
6554593,ORPHA_544472,Atypical hemolytic uremic syndrome with comple...,0.996973
3495385,ORPHA_98827,Unclassified myelodysplastic syndrome,0.996880
1768328,ORPHA_859,Transcobalamin deficiency,0.996801
2256176,ORPHA_488,Urachal cyst,0.996656
5067148,ORPHA_98870,Congenital dyserythropoietic anemia type III,0.996503


## 8. Export Results

In [16]:
# Export query results to CSV
# Uncomment and modify as needed

# output_file = '../temp/my_query_results.csv'
# top_results.to_csv(output_file, index=False)
# print(f'Results exported to: {output_file}')

## Notes

### Interpreting Prediction Scores

- **High (>0.8)**: Strong predicted therapeutic transferability between diseases
- **Medium (0.5-0.8)**: Moderate association, may warrant further investigation
- **Low (<0.5)**: Weak or no predicted association

### Model Performance

- **AUROC**: 0.89 on IxIDN/ORDON benchmarks (cross-evidence generalization)
- **Training**: MechDist network (disease-disease similarities from GO/HPO)
- **Evaluation**: Investment Decision Network (IxIDN) from clinical trials

### Coverage

- **Total diseases**: 4,710 (4,565 rare + 145 non-rare)
- **Total predictions**: ~22 million disease pairs

### Citation

If you use these predictions, please cite:

Ravandi et al. (2026). "Representation Learning of Human Diseases for Indication Expansion and Investment Decisions." In Preparation.